In [0]:
# Define the secret scope name for storing sensitive credentials
# This scope will contain Kafka connection details and API keys
secret_scope_name = "fraudradar-scope"

In [0]:
# Retrieve Databricks workspace API URL and authentication token from notebook context
# These credentials are used to make REST API calls to Databricks Secrets API

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url  = ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session

print(api_url)
print(api_token)  # handle securely, do not log in real code


import requests
import json

# ----------------------------------------
# Configuration
# ----------------------------------------
DATABRICKS_INSTANCE = api_url  # Replace with your workspace URL
DATABRICKS_TOKEN = api_token  # Replace with your PAT

scope_name = secret_scope_name  # Scope to be created
backend_type = "DATABRICKS"     # Use "AZURE_KEYVAULT" if integrating with Key Vault


In [0]:
# ----------------------------------------
# Create Secret Scope via Databricks REST API
# Backend type: DATABRICKS (managed by Databricks, not Azure Key Vault)
# ----------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope_name
}

# ----------------------------------------
# Send request
# ----------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# ----------------------------------------
# Handle response
# ----------------------------------------
if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
else:
    print("Failed to create secret scope.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)


In [0]:
# Define Kafka connection details for Confluent Cloud
# These will be stored securely in Databricks secrets (not hardcoded in production code)
kafka_bootstrap_servers='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
kafka_api_key='G3JBBMJJCOVKMBTE'
kafka_api_secret='cfltaJ/I+NTkTBUXyNTbt/X+aS7gQYMfIM7+IPT9NNI6vOr3U5zR4edO1deplEmg'
kafka_topic_name='credit_card_transactions'

# Serialize connection details to JSON for storage as a single secret
kafka_connection_details = json.dumps({
    "bootstrap_servers": kafka_bootstrap_servers,
    "topic": kafka_topic_name,
    "api_key": kafka_api_key,
    "api_secret": kafka_api_secret
})

In [0]:
# Display the JSON string to verify structure before storing as secret
print(kafka_connection_details)

In [0]:
# Store Kafka connection details as a secret in the fraudradar-scope
# Uses Databricks Secrets API to securely persist the JSON configuration
import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key='kafka_connection_details'
secret_value=kafka_connection_details

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)

In [0]:
# Verify that the Kafka connection details secret was stored correctly
# Retrieves the secret using dbutils and parses the JSON to validate structure
try:
    retrieved_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key='kafka_connection_details'
    )
    
    print("Secret retrieved successfully.")
    
    parsed = json.loads(retrieved_json)
    print("Parsed JSON:")
    print(parsed)
    
except Exception as e:
    print("Secret verification failed:")
    print(str(e))

In [0]:
# Store Gmail API application password as a secret for email notifications
# This is a Gmail app-specific password (not the main account password)
import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key = 'gmail_api_key'       # Name of the secret entry
secret_value = 'uhul dzjt wvfe rcvv' # Value to store securely

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)